In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

train_data = pd.read_csv('./data/train.csv') # Importing training data

new_train_data = pd.read_csv("./data/train_new.csv")

train_data = pd.concat((train_data, new_train_data), axis = 1, join = "inner")

train_data["P"] = train_data["P"].fillna(0)
train_data["O"] = train_data["O"].fillna(train_data["O"].median())

X_train = train_data.drop(["time", "Y1", "Y2"], axis = 1) # Losing the time and target columns

# Setting target variables
y1 = train_data["Y1"]

y2 = train_data["Y2"]

In [ ]:
from sklearn.decomposition import PCA

# Applying PCA to training set
X_train_scaled = (X_train - X_train.mean(axis=0)) / X_train.std(axis=0)

pca = PCA(n_components= 3)

X_pca = pca.fit_transform(X_train_scaled)

X_train["PC1"] = X_pca[:,0]
X_train["PC2"] = X_pca[:,1]
X_train["PC3"] = X_pca[:,2]

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns = ["PC1", "PC2", "PC3"],
    index = X_train.drop(["PC1", "PC2", "PC3"], axis = 1).columns
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate

# Choosing Features
X1_train = X_train[["G", "M", "J", "C", "E", "H", "N", "PC1"]]

# Modelling and Evaluation
modelY1 = LinearRegression()

cvY1 = cross_validate(
    modelY1,
    X1_train, y1,
    scoring = "r2",
    cv = 100
)

# Choosing Features

X2_train = X_train

# Modelling
modelY2 = RandomForestRegressor(n_estimators=219, n_jobs = -1, ccp_alpha=0, criterion="squared_error", max_depth=23, max_features="log2", max_leaf_nodes=None, max_samples=None, min_impurity_decrease=0, min_samples_leaf=3, min_samples_split=4,
                                min_weight_fraction_leaf=0, random_state=42, verbose=0, warm_start=False)

cvY2 = cross_validate(
    modelY2,
    X2_train, y2,
    scoring = "r2",
    cv = 5
)


# OUTPUTS
print(f"Predicted score : {(cvY2["test_score"].mean() + cvY1["test_score"].mean())/2}")
print(f"Score Y1 : {cvY1["test_score"].mean()}\nScore Y2 : {cvY2["test_score"].mean() }")

In [ ]:
modelY1.fit(X1_train, y1)
modelY2.fit(X2_train, y2)

In [ ]:

from sklearn.decomposition import PCA

test_data = pd.read_csv('./data/test.csv') # Importing testing data
new_test_data = pd.read_csv('./data/test_new.csv') # Importing new testing data

test_data = pd.concat((test_data, new_test_data), axis = 1, join = "inner")

test_data["P"] = test_data["P"].fillna(0)
test_data["O"] = test_data["O"].fillna(test_data["O"].median())

X = test_data.drop(["time", "id"], axis = 1) # Losing the time column

# Applying PCA to testing set
X_test = (X - X.mean(axis=0)) / X.std(axis=0)

X_pca = X_test.dot(loadings)

X = pd.concat((X, X_pca), axis = 1, join = "inner")

X1 = X[["G", "M", "J", "C", "E", "H", "N", "PC1"]]

y1_pred = pd.DataFrame(modelY1.predict(X1), index = test_data.id, columns = ["Y1"])

X2 = X

y2_pred = pd.DataFrame(modelY2.predict(X2), index = test_data.id, columns = ["Y2"])

out = pd.concat((y1_pred, y2_pred), axis = 1)

out.to_csv("./data/predictions.csv") # Scores 0.699

In [ ]:
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import r2_score
# from sklearn.linear_model import LinearRegression
# from scipy.stats import randint
# from sklearn.model_selection import RandomizedSearchCV

# # Choosing Features
# X1_train = X_train[["G", "M", "J", "C", "E", "H", "N", "PC1"]]
# X1_val = X_val[["G", "M", "J", "C", "E", "H", "N", "PC1"]]

# # Modelling
# modelY1 = LinearRegression()

# modelY1.fit(X1_train, y1_train)

# # Predicting
# y1_pred = modelY1.predict(X1_val)

# # Choosing Features

# X2_train = X_train[["A" ,"PC2", "K", "B", "D", "F", "I", "K", "L"]]
# X2_val = X_val[["A" ,"PC2", "K", "B", "D", "F", "I", "K", "L"]]

# # Modelling for y2
# param_dist = {
#     'n_estimators': randint(100, 150),'max_depth': randint(5, 30),'min_samples_split': randint(2, 15),'min_samples_leaf': randint(1, 10),'max_features': ['sqrt', 'log2', None],'bootstrap': [True, False]
# }

# # Base model
# rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# # RandomizedSearchCV
# rf_search = RandomizedSearchCV(
#     estimator=rf,
#     param_distributions=param_dist,
#     n_iter=20,
#     cv=3,
#     scoring='r2',
#     verbose=1,
#     n_jobs=-1,
#     random_state=42
# )


# rf_search.fit(X2_train, y2_train)


# best_rf_model = rf_search.best_estimator_


# y2_pred = best_rf_model.predict(X2_val)

# print(f"Predicted score : {(r2_score(y2_pred, y2_val) + r2_score(y1_pred, y1_val))/2}")
# print(f"Score Y1 : {r2_score(y1_pred, y1_val)}\nScore Y2 : {r2_score(y2_pred, y2_val)}")